# Импорт данных из CSV в базу данных

Этот скрипт импортирует данные из CSV-файла в базу данных с новой структурой.
Вклад авторов устанавливается в 0, затем будет указан вручную.

In [1]:
import pandas as pd
import psycopg2
import numpy as np
from datetime import datetime
import re

In [2]:
# Функция подключения к базе данных
def connect_to_db():
    try:
        conn = psycopg2.connect(
            dbname="postgres",
            user="postgres",
            password="123",
            host="localhost",
            port="5432"
        )
        return conn
    except Exception as e:
        print(f"Ошибка подключения к базе данных: {e}")
        return None

# Установка соединения
conn = connect_to_db()
if conn is None:
    raise Exception("Не удалось подключиться к базе данных")

# Установка схемы по умолчанию
cursor = conn.cursor()
cursor.execute("SET search_path TO elibrary")
conn.commit()

In [3]:
# Загрузка данных из CSV файла
# Замените 'your_file.csv' на путь к вашему CSV-файлу
df = pd.read_csv('final_result.csv', sep=',', encoding='utf-8')

# Просмотр первых строк для понимания структуры
print(df.head())
print(df.columns.tolist())

         ID                                           Название  \
0  80340461        СХЕМОТЕХНИКА ТЕЛЕКОММУНИКАЦИОННЫХ УСТРОЙСТВ   
1  80340463                             ЭЛЕКТРОННЫЕ УСТРОЙСТВА   
2  80490396  ТЕОРЕТИЧЕСКИЕ ОСНОВЫ ФИЗИЧЕСКОЙ ПОДГОТОВКИ СЛУ...   
3  80641447      СОВРЕМЕННЫЕ ТЕНДЕНЦИИ ИНЖЕНЕРНОГО ОБРАЗОВАНИЯ   
4  80528783  СПОСОБ ПРОЕКТИРОВАНИЯ СТРУКТУРЫ ВОЛОКОННО-ОПТИ...   

                                              Авторы  Год публикации В_РИНЦ  
0              Редкова Н.А., Кузина Е.И., Кузин П.И.          2025.0     Да  
1  Базир Г.И., Иванов С.Е., Редкова Н.А., Кузина ...          2025.0     Да  
2  Федюк Н.С., Федоров А.В., Марин А.П., Андросов...          2025.0     Да  
3                                        Нет авторов          2025.0     Да  
4  Горай И.И., Журавлёв Д.А., Калайтанова Е.В., М...          2025.0     Да  
['ID', 'Название', 'Авторы', 'Год публикации', 'В_РИНЦ']


In [4]:
# Функция для извлечения имен авторов из строки
def extract_authors(authors_str):
    if pd.isna(authors_str) or authors_str == 'Нет авторов':
        return []
    
    # Разделяем строку по запятым, но учитываем, что инициалы могут содержать точки
    # Например: "Редкова Н.А., Кузина Е.И., Кузин П.И."
    authors = []
    
    # Регулярное выражение для поиска имен авторов в формате Фамилия И.О.
    pattern = r'[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+[А-ЯЁ]\.[А-ЯЁ]?\.'
    matches = re.findall(pattern, authors_str)
    
    for match in matches:
        authors.append(match.strip())
    
    return authors

In [5]:
# Вспомогательная функция для получения ID пользователя по ФИО
def get_user_id_by_name(full_name):
    # Поиск пользователя по полному имени (full_name)
    query = f"SELECT id FROM users WHERE full_name ILIKE %s LIMIT 1"
    cursor.execute(query, (f'%{full_name}%',))
    result = cursor.fetchall()
    if result:
        return result[0][0]
    else:
        return None

In [6]:
# Обработка и добавление статей в базу данных
successful_imports = 0
failed_imports = 0

for index, row in df.iterrows():
    print(f"Обработка строки {index + 1}: ID {row['ID']}")
    
    try:
        # Проверяем, существует ли уже статья с таким external_id
        check_query = "SELECT id FROM articles WHERE external_id = %s"
        cursor.execute(check_query, (row['ID'],))
        existing_article = cursor.fetchall()
        
        if existing_article:
            print(f"  Статья с external_id {row['ID']} уже существует, пропускаем...")
            continue
        
        # Определяем значение in_rinc
        in_rinc = True if row['В_РИНЦ'] == 'Да' else False
        
        # Добавляем статью
        insert_article_query = """
        INSERT INTO articles (external_id, title, year_pub, in_rinc)
        VALUES (%s, %s, %s, %s)
        RETURNING id;
        """
        
        cursor.execute(insert_article_query, (row['ID'], row['Название'], row['Год публикации'], in_rinc))
        article_id = cursor.fetchone()[0]
        conn.commit()
        
        print(f"  Добавлена статья с ID {article_id}")
        
        # Обработка авторов
        authors = extract_authors(row['Авторы'])
        
        if len(authors) == 0:
            print(f"  Нет авторов для статьи {row['ID']}")
        else:
            for author_name in authors:
                # Устанавливаем вклад автора в 0, затем будет указан вручную
                contribution = 0.0
                
                # Проверяем, является ли автор сотрудником
                user_employee_id = get_user_id_by_name(author_name)
                
                # Формируем SQL-запрос для добавления автора
                if user_employee_id:
                    # Автор является сотрудником
                    insert_author_query = """
                    INSERT INTO authors (article_id, author_name, user_employee_id, contribution, applied_for_award, award_applied_date)
                    VALUES (%s, %s, %s, %s, %s, %s)
                    """
                    cursor.execute(insert_author_query, (article_id, author_name, user_employee_id, contribution, False, None))
                else:
                    # Автор не является сотрудником
                    insert_author_query = """
                    INSERT INTO authors (article_id, author_name, user_employee_id, contribution, applied_for_award, award_applied_date)
                    VALUES (%s, %s, NULL, %s, %s, %s)
                    """
                    cursor.execute(insert_author_query, (article_id, author_name, contribution, False, None))
                
                conn.commit()
                
                print(f"    Добавлен автор: {author_name}, вклад: {contribution}%, сотрудник ID: {user_employee_id}")
        
        successful_imports += 1
        
    except Exception as e:
        print(f"  Ошибка при обработке строки {index + 1}: {str(e)}")
        conn.rollback()  # Откатываем транзакцию в случае ошибки
        failed_imports += 1

print(f"\nИмпорт завершен. Успешно: {successful_imports}, Ошибок: {failed_imports}")

Обработка строки 1: ID 80340461
  Добавлена статья с ID 1
    Добавлен автор: Редкова Н.А., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Кузина Е.И., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Кузин П.И., вклад: 0.0%, сотрудник ID: None
Обработка строки 2: ID 80340463
  Добавлена статья с ID 2
    Добавлен автор: Базир Г.И., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Иванов С.Е., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Редкова Н.А., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Кузина Е.И., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Кузин П.И., вклад: 0.0%, сотрудник ID: None
Обработка строки 3: ID 80490396
  Добавлена статья с ID 3
    Добавлен автор: Федюк Н.С., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Федоров А.В., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Марин А.П., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Андросов А.М., вклад: 0.0%, сотрудник ID: None
    Добавлен автор: Буздов А.Ю., вклад: 0.0%, 

In [7]:
# Проверка добавленных данных
print("Количество добавленных статей:")
count_query = "SELECT COUNT(*) as count FROM articles WHERE external_id IS NOT NULL"
cursor.execute(count_query)
result = cursor.fetchone()
print(result[0])

print("\nПример добавленных статей:")
sample_query = "SELECT * FROM articles WHERE external_id IS NOT NULL ORDER BY external_id DESC LIMIT 5"
cursor.execute(sample_query)
sample_articles = cursor.fetchall()
for article in sample_articles:
    print(article)

print("\nПример авторов:")
sample_authors_query = "SELECT * FROM authors LIMIT 5"
cursor.execute(sample_authors_query)
sample_authors = cursor.fetchall()
for author in sample_authors:
    print(author)

# Закрываем соединение
conn.close()

Количество добавленных статей:
12655

Пример добавленных статей:
(323, 82445720, 'ИНОСТРАННЫЕ ЯЗЫКИ В ОБРАЗОВАТЕЛЬНЫХ ОРГАНИЗАЦИЯХ МИНОБОРОНЫ РОССИИ: ЧАСТНЫЕ МЕТОДИКИ И УЧЕБНЫЕ МАТЕРИАЛЫ', 2024, False)
(169, 82436724, 'ПРОВЕДЕНИЕ АНАЛИЗА УЯЗВИМОСТЕЙ И ПОТЕНЦИАЛЬНЫХ УГРОЗ ИНФОРМАЦИОННОЙ БЕЗОПАСНОСТИ ПРИ ИСПОЛЬЗОВАНИИ ОБЛАЧНЫХ СЕРВИСОВ', 2025, False)
(163, 82436723, 'ИСКУССТВЕННЫЙ ИНТЕЛЛЕКТ В ОБРАЗОВАНИИ КУРСАНТОВ-СВЯЗИСТОВ: ПЕРСПЕКТИВЫ И ДОСТИЖЕНИЯ', 2025, False)
(139, 82434476, 'ЭТИЧЕСКАЯ СОСТАВЛЯЮЩАЯ КУЛЬТУРЫ БЕЗОПАСНОСТИ СОЦИУМА', 2025, True)
(284, 82429537, 'МЕТОДИЧЕСКОЕ И ПСИХОЛОГИЧЕСКОЕ СОПРОВОЖДЕНИЕ СТУДЕНТОВ, ЗАНИМАЮЩИХСЯ В ГРУППЕ ЛФК', 2025, False)

Пример авторов:
(1, 1, 'Редкова Н.А.', None, 0.0, False, None)
(2, 1, 'Кузина Е.И.', None, 0.0, False, None)
(3, 1, 'Кузин П.И.', None, 0.0, False, None)
(4, 2, 'Базир Г.И.', None, 0.0, False, None)
(5, 2, 'Иванов С.Е.', None, 0.0, False, None)


## Инструкции по использованию

1. Замените `'your_file.csv'` в четвертой ячейке на путь к вашему CSV-файлу
2. Убедитесь, что столбцы в файле имеют названия: `ID`, `Название`, `Авторы`, `Год публикации`, `В_РИНЦ`
3. Запустите все ячейки по порядку

Скрипт автоматически:
- Проверит, не существует ли уже статья с таким ID
- Добавит статью с external_id
- Разберет список авторов
- Установит вклад авторов в 0 (для последующего ручного указания)
- Свяжет авторов с сотрудниками, если такие существуют в БД
- Установит флаг in_rinc в зависимости от значения в CSV